# Experiment 9 stimulus report

Inspect generated RF-memory stimuli as angle/dot-size traces, spatial coverage, and explicit preconditioning/delay/bout timing. No initial static display is included. Spatial coverage excludes OFF frames; dot counts refer to recorded identities.

Generate the assets first with `python scripts/stimuli/generation/Trajectory_exp_9_generator.py`. Edit the JSON configurations to change stimulus parameters. This notebook reads the archived resolved settings and saves previews beside the generated trajectories.


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'src' / 'stimulus_visualization.py').is_file())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.stimulus_visualization import (
    load_stimulus_trajectories, summarize_stimuli,
    plot_stimulus_traces, plot_stimulus_positions,
    plot_stimulus_phase_timing,
)

In [ ]:
EXPERIMENT = 'Exp_9_RF_memory_inhibition'
DERIVED_ROOT = Path.home() / 'OneDrive - Université de Lausanne/Lab/Results/Paper/2p_derived'
INPUT_DIR = DERIVED_ROOT / EXPERIMENT / 'stimuli'
PATTERN = '*_trajectory.csv'

settings = json.loads((INPUT_DIR / 'parameters/resolved_settings.json').read_text(encoding='utf-8'))
rates = {s['framerate'] for s in settings.values()}
rotations = {s['rotation_angle_deg'] for s in settings.values()}
if len(rates) != 1 or len(rotations) != 1:
    raise ValueError('Inspect hemifields separately when their frame rates or rotations differ.')


In [ ]:
samples = load_stimulus_trajectories(
    INPUT_DIR, pattern=PATTERN, framerate=next(iter(rates)),
    rotation_angle_deg=next(iter(rotations)),
)
summary = summarize_stimuli(samples)
timings = pd.read_csv(INPUT_DIR / 'parameters/stimulus_time_summary.csv')


## Standard representation
Solid lines show angle; dashed lines show the CSV dot size, including off frames. Colors identify dots. Size is not distance from the fish.

In [ ]:
standard_figure = plot_stimulus_traces(samples, cols=3, panel_size=(6, 3.8))
display(standard_figure)
plt.close(standard_figure)

## Spatial coverage
Each panel shows the unique visible directions for one stimulus, with equal-size markers on a circle and the fish at the center. The circle does not encode distance or dot size.

In [ ]:
polar_figure = plot_stimulus_positions(summary, cols=4, panel_size=(3.8, 3.8))
display(polar_figure)
plt.close(polar_figure)

## Preconditioning, blank delay, and test bout

Colored bars show the explicit generated phases; black pulses mark frames when any dot is visible. Delays begin at the last visible flicker offset. External before/after pauses are excluded. This view uses the generator's phase metadata rather than inferring bout onset from movement.


In [ ]:
timing_figure = plot_stimulus_phase_timing(samples, timings, figsize=(12, 8))
display(timing_figure)
plt.close(timing_figure)


## Summary and automatic export

The final cell saves only the spatial and timing previews. Angle/dot-size traces and the summary remain available in the notebook. The generator saves the short stimulus specification in `experiment_9_report.md`, with complete timings in `experiment_9_timing.xlsx`.


In [ ]:
display(summary)
polar_figure.savefig(INPUT_DIR / 'stimuli_polar.png', dpi=300)
timing_figure.savefig(INPUT_DIR / 'stimuli_timing.png', dpi=300)
print(f'Saved spatial and timing previews to {INPUT_DIR}')
